# 02. Feature Extraction
Convert quantum states into a 15-dimensional Pauli expectation feature matrix.

In [26]:
pip install qiskit numpy pandas matplotlib

In [27]:
import numpy as np
import pandas as pd

import itertools
from qiskit.quantum_info import Statevector, Pauli

In [28]:
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR = "/content/drive/MyDrive/entanglement-detector/data"
df = pd.read_csv(f"{DATA_DIR}/labels.csv")
X_states = np.load(f"{DATA_DIR}/states.npy")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Measurement combinations

In [29]:
PAULIS = ["I", "X", "Y", "Z"]
MEASUREMENTS = [f"{a}{b}" for a, b in itertools.product(PAULIS, PAULIS)][1:]

print(f"Number of measurements: {len(MEASUREMENTS)}")
print(MEASUREMENTS)

Number of measurements: 15
['IX', 'IY', 'IZ', 'XI', 'XX', 'XY', 'XZ', 'YI', 'YX', 'YY', 'YZ', 'ZI', 'ZX', 'ZY', 'ZZ']


## 3. Feature extraction

In [30]:
def extract_features(amplitudes):
    """Measure a state in 15 directions -> 15 numbers."""
    state = Statevector(amplitudes)
    return [np.real(state.expectation_value(Pauli(m))) for m in MEASUREMENTS]

## 4. Build the feature matrix

In [31]:
X = np.array([extract_features(v) for v in X_states])
y = df["label"].values

print("--- Results ---")
print(f"Feature matrix shape : {X.shape}")
print(f"Labels shape         : {y.shape}")
print(f"Value range          : {X.min():.4f} to {X.max():.4f}")

--- Results ---
Feature matrix shape : (5000, 15)
Labels shape         : (5000,)
Value range          : -0.9998 to 1.0000


## 5. Inspect as a table

In [32]:
features_df = pd.DataFrame(X, columns=MEASUREMENTS)
features_df["label"] = y
features_df.head()

,IX,IY,IZ,XI,XX,XY,XZ,YI,YX,YY,YZ,ZI,ZX,ZY,ZZ,label
0,-0.034162,-0.843002,0.536824,-0.198890,0.006794,0.167665,-0.106769,0.557382,-0.019041,-0.469875,0.299216,0.806082,-0.027537,-0.679529,0.432724,0
1,0.268999,-0.127543,0.954658,0.999882,0.268967,-0.127528,0.954545,0.015065,0.004052,-0.001921,0.014382,0.003135,0.000843,-0.000400,0.002993,0
2,0.042576,0.804668,0.592196,0.377614,0.016077,0.303854,0.223622,-0.875364,-0.037270,-0.704377,-0.518387,-0.301904,-0.012854,-0.242933,-0.178787,0
3,-0.687473,-0.588401,-0.425634,-0.781796,0.537464,0.460010,0.332759,-0.051404,0.035339,0.030246,0.021879,-0.621411,0.427204,0.365639,0.264494,0
4,-0.446072,0.893623,-0.049571,0.193876,-0.086483,0.173252,-0.009611,-0.823505,0.367342,-0.735903,0.040822,0.533153,-0.237825,0.476438,-0.026429,0


## 6. Save

In [33]:
np.save(f"{DATA_DIR}/X_features.npy", X)
np.save(f"{DATA_DIR}/y_labels.npy", y)

print("--- Results ---")
print(f"Saved features: {X.shape}")

--- Results ---
Saved features: (5000, 15)


## 7. Analysis

Each state is now described by 15 real numbers in the range -1 to 1, corresponding to expectation values of two-qubit Pauli measurements. This converts the quantum states into a standard tabular classification problem with 15 features and a binary target.